# 15_E7 - Al-Kafri mask decode patch

Este patch corrige el problema detectado en `FAST visual sanity`: las máscaras PNG no deben binarizarse con `arr > 0`, porque muchas tienen fondo no-cero.

La estrategia nueva estima el fondo por color/valor dominante en el borde de la imagen y define foreground como los píxeles distintos al fondo.


In [ ]:
import importlib.util, subprocess, sys
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])
ensure_package('pydicom')
ensure_package('skimage', 'scikit-image')

import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.transform import resize
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive no montado automáticamente:', repr(exc))

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 140)


In [ ]:
def find_pfi_root():
    expected = Path('/content/drive/MyDrive/PFI_MVP')
    marker = expected / 'results/E7_alkafri_axial_curated_subset/E7_alkafri_axial_strict_candidate_pairs_FAST.csv'
    if marker.exists():
        return expected
    for base in [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')]:
        if not base.exists():
            continue
        matches = list(base.rglob('E7_alkafri_axial_strict_candidate_pairs_FAST.csv'))
        if matches:
            parts = matches[0].parts
            if 'results' in parts:
                return Path(*parts[:parts.index('results')])
    return expected

PFI_ROOT = find_pfi_root()
CURATION_ROOT = PFI_ROOT / 'results' / 'E7_alkafri_axial_curated_subset'
FIGURES_ROOT = PFI_ROOT / 'figures'
DOCS_ROOT = PFI_ROOT / 'docs'
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
DOCS_ROOT.mkdir(parents=True, exist_ok=True)

fast_candidates_path = CURATION_ROOT / 'E7_alkafri_axial_strict_candidate_pairs_FAST.csv'
if not fast_candidates_path.exists():
    raise FileNotFoundError(f'No existe {fast_candidates_path}. Primero correr fast_candidates_patch.')

fast_df = pd.read_csv(fast_candidates_path, low_memory=False)
print('PFI_ROOT:', PFI_ROOT)
print('FAST candidates:', fast_df.shape)
display(fast_df.head())

MAX_VIS_DECODED = 80
RANDOM_SEED = 42


In [ ]:
def read_dicom_pixels(path):
    ds = pydicom.dcmread(str(path), force=True)
    return ds.pixel_array.astype(np.float32), ds

def normalize_image(arr):
    arr = np.asarray(arr, dtype=np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr, dtype=np.float32)
    arr = np.clip(arr, p1, p99)
    return (arr - p1) / (p99 - p1 + 1e-8)

def dominant_row_value(values):
    values = np.asarray(values)
    if values.ndim == 1:
        uniq, counts = np.unique(values, return_counts=True)
        return uniq[np.argmax(counts)]
    flat = values.reshape(-1, values.shape[-1])
    uniq, counts = np.unique(flat, axis=0, return_counts=True)
    return uniq[np.argmax(counts)]

def border_pixels(arr, border=8):
    h, w = arr.shape[:2]
    b = max(1, min(border, h // 4, w // 4))
    top = arr[:b, ...]
    bottom = arr[-b:, ...]
    left = arr[:, :b, ...]
    right = arr[:, -b:, ...]
    return np.concatenate([top.reshape(-1, *arr.shape[2:]), bottom.reshape(-1, *arr.shape[2:]), left.reshape(-1, *arr.shape[2:]), right.reshape(-1, *arr.shape[2:])], axis=0)

def decode_mask_foreground(path, tolerance=3):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    if arr.ndim == 3:
        bg = dominant_row_value(border_pixels(arr))
        diff = np.max(np.abs(arr.astype(np.int16) - bg.astype(np.int16)), axis=-1)
        mask = (diff > tolerance).astype(np.uint8)
        # Fallback: si el borde no representa bien el fondo, usar color dominante global.
        ratio = float(mask.mean())
        if ratio > 0.80 or ratio == 0.0:
            bg2 = dominant_row_value(arr)
            diff2 = np.max(np.abs(arr.astype(np.int16) - bg2.astype(np.int16)), axis=-1)
            mask2 = (diff2 > tolerance).astype(np.uint8)
            if 0.0 < mask2.mean() < ratio:
                bg, mask = bg2, mask2
        bg_repr = tuple(int(x) for x in np.asarray(bg).tolist())
        return mask, arr, bg_repr
    else:
        bg = dominant_row_value(border_pixels(arr))
        mask = (np.abs(arr.astype(np.int16) - int(bg)) > tolerance).astype(np.uint8)
        ratio = float(mask.mean())
        if ratio > 0.80 or ratio == 0.0:
            bg2 = dominant_row_value(arr)
            mask2 = (np.abs(arr.astype(np.int16) - int(bg2)) > tolerance).astype(np.uint8)
            if 0.0 < mask2.mean() < ratio:
                bg, mask = bg2, mask2
        return mask, arr, int(bg)

def resize_mask_for_display(mask, target_shape):
    if tuple(mask.shape) == tuple(target_shape):
        return mask
    resized = resize(mask.astype(np.float32), target_shape, order=0, preserve_range=True, anti_aliasing=False)
    return (resized > 0.5).astype(np.uint8)

def mask_sanity(mask):
    fg = mask > 0
    ratio = float(fg.mean())
    _, ncomp = ndimage.label(fg)
    return {
        'mask_empty': bool(ratio == 0.0),
        'mask_full': bool(ratio > 0.95),
        'foreground_ratio': ratio,
        'component_count': int(ncomp),
    }

# Diagnóstico rápido de colores/valores en primeras máscaras.
diagnostic_rows = []
for i, (_, row) in enumerate(fast_df.head(12).iterrows(), start=1):
    mask, raw, bg = decode_mask_foreground(row['gt_file_path'])
    diagnostic_rows.append({
        'i': i,
        'candidate_id': f'decode_diag_{i:03d}',
        'gt_file_path': row['gt_file_path'],
        'raw_shape': str(raw.shape),
        'decoded_bg': str(bg),
        'decoded_foreground_ratio': float(mask.mean()),
        'decoded_unique_values': int(len(np.unique(mask))),
    })
diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(CURATION_ROOT / 'E7_alkafri_mask_decode_diagnostic.csv', index=False)
display(diagnostic_df)


In [ ]:
# Selección visual: priorizar T1 y final/intermediary, pero mantener variedad.
vis_df = fast_df.copy()
vis_df['modality_priority'] = vis_df['modality'].map({'T1': 0, 'T2': 1}).fillna(9)
vis_df['gt_type_priority'] = vis_df['gt_type'].map({'final': 0, 'intermediary': 1}).fillna(9)
vis_df = vis_df.sort_values(['modality_priority', 'gt_type_priority', 'case_id', 'disc_or_slice_id', 'instance_number']).head(MAX_VIS_DECODED).copy()

figure_rows = []
sanity_rows = []
print('Candidatos para visualización DECODED:', len(vis_df))

for i, (_, row) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc='decoded visual sanity'), start=1):
    candidate_id = f'decoded_candidate_{i:03d}'
    image_path = row['image_file_path']
    gt_path = row['gt_file_path']
    sanity = {
        'candidate_id': candidate_id,
        'image_file_path': image_path,
        'gt_file_path': gt_path,
        'case_id': row.get('case_id'),
        'modality': row.get('modality'),
        'gt_type': row.get('gt_type'),
        'disc_or_slice_id': row.get('disc_or_slice_id'),
        'instance_number': row.get('instance_number'),
        'overlay_generated': False,
        'auto_sanity_status': 'error',
    }
    try:
        img, _ = read_dicom_pixels(image_path)
        img_norm = normalize_image(img)
        mask, raw_mask, decoded_bg = decode_mask_foreground(gt_path)
        mask_for_display = resize_mask_for_display(mask, img.shape)
        ms = mask_sanity(mask_for_display)
        sanity.update(ms)
        sanity['decoded_background'] = str(decoded_bg)
        sanity['image_shape_actual'] = str(tuple(img.shape))
        sanity['mask_shape_actual'] = str(tuple(mask.shape))
        sanity['raw_mask_shape_actual'] = str(tuple(raw_mask.shape))
        sanity['resize_needed'] = bool(tuple(mask.shape) != tuple(img.shape))
        sanity['overlay_generated'] = True
        auto_ok = (not ms['mask_empty'] and not ms['mask_full'] and 0.0005 <= ms['foreground_ratio'] <= 0.70)
        sanity['auto_sanity_status'] = 'ok' if auto_ok else 'review'

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(img_norm, cmap='gray'); axes[0].set_title('Axial IMA'); axes[0].axis('off')
        axes[1].imshow(raw_mask if raw_mask.ndim == 3 else raw_mask, cmap=None if raw_mask.ndim == 3 else 'gray'); axes[1].set_title('Raw GT PNG'); axes[1].axis('off')
        axes[2].imshow(mask_for_display, cmap='gray'); axes[2].set_title(f'Decoded mask\nfg={ms["foreground_ratio"]:.3f}'); axes[2].axis('off')
        axes[3].imshow(img_norm, cmap='gray')
        axes[3].imshow(np.ma.masked_where(mask_for_display <= 0, mask_for_display), alpha=0.45, cmap='autumn')
        axes[3].set_title('Overlay decoded'); axes[3].axis('off')
        fig.suptitle(f"{candidate_id} | case={row.get('case_id')} | {row.get('modality')} | gt={row.get('gt_type')} | D={row.get('disc_or_slice_id')} | inst={row.get('instance_number')} | {sanity['auto_sanity_status']}", fontsize=10)
        fig.tight_layout()
        fig_path = FIGURES_ROOT / f'E7_alkafri_decoded_curated_candidate_{i:03d}.png'
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        figure_rows.append({
            'candidate_id': candidate_id,
            'figure_path': str(fig_path),
            'image_file_path': image_path,
            'gt_file_path': gt_path,
            'case_id': row.get('case_id'),
            'modality': row.get('modality'),
            'gt_type': row.get('gt_type'),
            'disc_or_slice_id': row.get('disc_or_slice_id'),
            'instance_number': row.get('instance_number'),
            'foreground_ratio': ms['foreground_ratio'],
            'auto_sanity_status': sanity['auto_sanity_status'],
            'manual_accept': '',
            'manual_reject_reason': '',
            'notes': '',
        })
    except Exception as exc:
        sanity['error'] = repr(exc)
    sanity_rows.append(sanity)

review_decoded_df = pd.DataFrame(figure_rows)
sanity_decoded_df = pd.DataFrame(sanity_rows)
review_path = CURATION_ROOT / 'E7_alkafri_axial_curation_review_sheet_DECODED.csv'
sanity_path = CURATION_ROOT / 'E7_alkafri_axial_candidate_sanity_checks_DECODED.csv'
review_decoded_df.to_csv(review_path, index=False)
sanity_decoded_df.to_csv(sanity_path, index=False)
print('Figuras DECODED generadas:', len(review_decoded_df))
print('Review sheet:', review_path)
print('Sanity checks:', sanity_path)
display(review_decoded_df.head())
display(sanity_decoded_df.head())

auto_ok = int((sanity_decoded_df['auto_sanity_status'] == 'ok').sum()) if len(sanity_decoded_df) else 0
report = {
    'mode': 'mask_decode_patch',
    'input_fast_candidates': int(len(fast_df)),
    'visual_candidates_decoded': int(len(review_decoded_df)),
    'auto_sanity_ok_decoded': auto_ok,
    'review_sheet': str(review_path),
    'sanity_checks': str(sanity_path),
    'recommended_next_step': 'open_decoded_figures_and_manually_accept_reliable_overlays',
}
report_path = CURATION_ROOT / 'E7_alkafri_axial_mask_decode_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
